# 🛠️ Day 02a: Feature Engineering — Make Your Features Do The Work

---

## 🎯 What You'll Master Today
- Polynomial & interaction features
- Binning, log/power transforms
- Target encoding
- Feature selection (mutual info, RFE, L1)

**Goal:** Improve model performance by engineering better features.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  FEATURE ENGINEERING — The 80/20 Rule                            ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  "Better features > better models"                               ║
║                                                                   ║
║  CREATE FEATURES:                                                ║
║    PolynomialFeatures → x², x₁·x₂                              ║
║    Log/Sqrt transform → reduce skewness                         ║
║    Binning            → continuous → categories                  ║
║    Interaction        → combine domain knowledge                 ║
║                                                                   ║
║  SELECT FEATURES:                                                ║
║    mutual_info_classif → non-linear dependency score            ║
║    RFE                → recursive elimination with model        ║
║    L1 regularization  → automatic selection via sparsity        ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures, KBinsDiscretizer, PowerTransformer
from sklearn.feature_selection import (
    mutual_info_classif, SelectKBest, RFE
)
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# ============================================================
# 1. Polynomial & Interaction Features
# ============================================================

X = np.array([[2, 3], [4, 5], [6, 7]])
print(f"Original shape: {X.shape}")

# degree=2 generates: [1, a, b, a², ab, b²]
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
print(f"Poly features: {poly.get_feature_names_out()}")
print(f"Shape: {X.shape} → {X_poly.shape}")
print(f"Row 0: {X[0]} → {X_poly[0]}")

# Interaction only (no x², x³) — often more useful
poly_int = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_int = poly_int.fit_transform(X)
print(f"\nInteraction only: {poly_int.get_feature_names_out()}")
print(f"Row 0: {X[0]} → {X_int[0]}")

print("\n💡 interaction_only=True often enough. Full poly can explode feature count.")

In [ ]:
# ============================================================
# 2. Log Transform & Power Transform
# ============================================================

# Skewed data — common in real world (income, prices, counts)
np.random.seed(42)
skewed = np.random.exponential(scale=2, size=1000)

# Log transform — the classic
log_transformed = np.log1p(skewed)  # log1p = log(1+x), safe for x=0
print(f"Original skew:  {pd.Series(skewed).skew():.3f}")
print(f"Log skew:       {pd.Series(log_transformed).skew():.3f}")

# PowerTransformer — automatic (Yeo-Johnson works with negatives)
pt = PowerTransformer(method='yeo-johnson')  # also 'box-cox' for positive-only
power_transformed = pt.fit_transform(skewed.reshape(-1, 1))
print(f"Power skew:     {pd.Series(power_transformed.flatten()).skew():.3f}")

print("\n💡 np.log1p() for manual. PowerTransformer for automatic.")
print("   Inverse: np.expm1() reverses log1p.")

In [ ]:
# ============================================================
# 3. Binning — Continuous → Categories
# ============================================================

ages = np.array([22, 35, 48, 19, 55, 67, 30, 42]).reshape(-1, 1)

# KBinsDiscretizer — sklearn's binning tool
kbd_uniform = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
binned_u = kbd_uniform.fit_transform(ages)
print(f"Uniform bins: {binned_u.flatten().astype(int)}")
print(f"  Edges: {kbd_uniform.bin_edges_[0].round(1)}")

kbd_quantile = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
binned_q = kbd_quantile.fit_transform(ages)
print(f"\nQuantile bins: {binned_q.flatten().astype(int)}")
print(f"  Edges: {kbd_quantile.bin_edges_[0].round(1)}")

# pd.cut — Pandas way (more control)
labels_cut = pd.cut(ages.flatten(), bins=[0, 25, 45, 100], labels=['young', 'mid', 'senior'])
print(f"\npd.cut: {list(labels_cut)}")

print("\n💡 strategy='quantile' for equal-sized bins. 'uniform' for equal-width.")

In [ ]:
# ============================================================
# 4. Feature Selection — Pick What Matters
# ============================================================

X, y = make_classification(
    n_samples=500, n_features=20, n_informative=5,
    n_redundant=5, n_useless=10, random_state=42
)
print(f"Original: {X.shape[1]} features (5 informative, 5 redundant, 10 useless)\n")

# --- Method 1: Mutual Information ---
mi_scores = mutual_info_classif(X, y, random_state=42)
top_mi = np.argsort(mi_scores)[::-1][:5]
print(f"Top 5 by Mutual Info: features {top_mi}")
print(f"  Scores: {mi_scores[top_mi].round(3)}\n")

# --- Method 2: RFE (Recursive Feature Elimination) ---
rfe = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=5)
rfe.fit(X, y)
selected_rfe = np.where(rfe.support_)[0]
print(f"RFE selected: features {selected_rfe}\n")

# --- Method 3: L1 Regularization (Lasso) ---
lasso = Lasso(alpha=0.01)
lasso.fit(X, y)
selected_l1 = np.where(np.abs(lasso.coef_) > 0.01)[0]
print(f"L1 selected: features {selected_l1}")
print(f"  (Non-zero coefficients: {len(selected_l1)})")

print("\n💡 MI = filter method (fast). RFE = wrapper (accurate). L1 = embedded (during training).")

In [ ]:
# ============================================================
# 5. Feature Importance from Trees
# ============================================================

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = rf.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print("Feature Importance (Random Forest):")
for i, idx in enumerate(sorted_idx[:8]):
    bar = '█' * int(importances[idx] * 50)
    print(f"  Feature {idx:2d}: {importances[idx]:.4f} {bar}")

# SelectKBest with mutual_info
selector = SelectKBest(score_func=mutual_info_classif, k=5)
X_selected = selector.fit_transform(X, y)
print(f"\nSelectKBest(k=5): {X.shape} → {X_selected.shape}")
print(f"Selected features: {np.where(selector.get_support())[0]}")

In [ ]:
# ============================================================
# 6. Real-World Feature Engineering Drill
# ============================================================

# Simulating a house price dataset
np.random.seed(42)
df = pd.DataFrame({
    'sqft': np.random.randint(500, 5000, 200),
    'bedrooms': np.random.randint(1, 6, 200),
    'bathrooms': np.random.randint(1, 4, 200),
    'year_built': np.random.randint(1960, 2023, 200),
    'lot_size': np.random.exponential(5000, 200),  # skewed!
})

# Domain-driven features
df['price_per_sqft'] = df['sqft'] / df['bedrooms']        # ratio feature
df['total_rooms'] = df['bedrooms'] + df['bathrooms']       # aggregation
df['house_age'] = 2024 - df['year_built']                  # time-based
df['log_lot_size'] = np.log1p(df['lot_size'])              # skew reduction
df['is_new'] = (df['year_built'] >= 2010).astype(int)      # binary flag

print("Engineered Features:")
print(df[['sqft', 'price_per_sqft', 'total_rooms', 'house_age', 'log_lot_size', 'is_new']].head())

print(f"\nLot size skew: {df['lot_size'].skew():.2f} → {df['log_lot_size'].skew():.2f} (log)")

print("\n💡 Feature engineering recipe:")
print("  1. Ratios (a/b)  2. Aggregations (a+b)  3. Time-based (age)")
print("  4. Log (skew fix)  5. Binary flags  6. Domain knowledge")

---

## 🗺️ Feature Engineering Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  FEATURE ENGINEERING — When To Use What                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  CREATE:                                                         ║
║    Polynomial      → capture non-linear relationships           ║
║    Log/Power       → fix skewed distributions                   ║
║    Binning         → continuous → discrete (age groups)         ║
║    Ratios/Agg      → domain knowledge (price/sqft)             ║
║    Time features   → year, month, day_of_week, is_weekend      ║
║                                                                   ║
║  SELECT:                                                         ║
║    Filter  → MI, chi², f_classif  (fast, model-agnostic)       ║
║    Wrapper → RFE, forward/backward  (accurate, slow)           ║
║    Embedded → L1/Lasso, tree importance  (during training)      ║
║                                                                   ║
║  RULE OF THUMB: Start with domain features, then tree import.   ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | How to handle skewed features? | Log transform (np.log1p), PowerTransformer (Yeo-Johnson/Box-Cox), or binning |
| 2 | Polynomial vs interaction features? | Polynomial: x², x³. Interaction: x₁·x₂. interaction_only=True avoids explosion |
| 3 | Filter vs wrapper vs embedded selection? | Filter: fast, model-agnostic (MI). Wrapper: model-based (RFE). Embedded: built-in (L1, trees) |
| 4 | Why use feature importance from trees? | Non-linear, handles interactions naturally, quick ranking of feature relevance |
| 5 | What is target encoding? | Replace category with mean of target. Risk: data leakage — must use CV-based or leave-one-out |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • log1p() for skew, PolynomialFeatures for non-linear │
## │  • Domain features (ratios, age, flags) beat auto-gen   │
## │  • MI = fast filter, RFE = accurate wrapper             │
## │  • Tree importance = quick feature ranking               │
## │  • Target encoding leaks — use CV-based variants        │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Model Selection** — cross_val_score, GridSearchCV, learning curves